In [3]:
import os, sys, time, json, math, random, uuid, re
from typing import List, Dict, Any, Tuple, Iterable
from collections import Counter, defaultdict
from pathlib import Path
import csv

SEED = 42
random.seed(SEED)

# =================== Config rápida ===================
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o")
EMBED_MODEL  = os.getenv("EMBED_MODEL", "text-embedding-3-small")
BP_MODE      = os.getenv("BP_MODE", "memory")

BP_VARIANTS_PER_CALL   = int(os.getenv("BP_VARIANTS_PER_CALL", "4"))
BP_MAX_CALLS_PER_ITEM  = int(os.getenv("BP_MAX_CALLS_PER_ITEM", "3"))
MIN_JACCARD_FOR_LABEL  = float(os.getenv("MIN_JACCARD_FOR_LABEL", "0.25"))
MAX_JACCARD_FOR_DIVERSITY = float(os.getenv("MAX_JACCARD_FOR_DIVERSITY", "0.92"))

FIXED_AUG_PER_ITEM = int(os.getenv("FIXED_AUG_PER_ITEM", "0"))

# NUEVO: Mínimo de muestras por clase
MIN_SAMPLES_PER_CLASS = int(os.getenv("MIN_SAMPLES_PER_CLASS", "200")) # <--- CAMBIO

INPUT_JSONL  = os.getenv("INPUT_JSONL",  "en_dataset_with_stop_words/en_dataset_with_stop_words_masked_train_low_regime.jsonl")
OUTPUT_JSONL = os.getenv("OUTPUT_JSONL", "en_dataset_with_stop_words/en_dataset_with_stop_words_masked_train_low_regime_augmented.jsonl")

# Sliding window
SW_WINDOW_SIZE = int(os.getenv("SW_WINDOW_SIZE", "3"))
SW_STRIDE      = int(os.getenv("SW_STRIDE", "3"))

# Memoria / RAG
USE_EMBEDDINGS         = os.getenv("USE_EMBEDDINGS", "1") not in ("0","false","False")
SIM_THRESHOLD          = float(os.getenv("SIM_THRESHOLD", "0.88"))
MAX_NEG_EXAMPLES       = int(os.getenv("MAX_NEG_EXAMPLES", "4"))
RAG_NEIGHBORS_PER_CAT  = int(os.getenv("RAG_NEIGHBORS_PER_CAT", "64"))

RETRIES      = 2
RETRY_SLEEP  = 1.0
BP_RETRIES   = 6
SEED_JITTER  = True

# =================== Taxonomía ===================
TAXONOMY = [
    "GENDER_WOMEN","GENDER_MEN","GENDER_NONBINARY","GENDER_IDENTITY_TRANS","GENDER_IDENTITY_OTHER",
    "RACE_BLACK","RACE_WHITE","RACE_OTHER",
    "ETH_ASIAN","ETH_LATINO","ETH_ARAB","ETH_JEWISH","ETH_ROMA","ETH_OTHER",
    "SEXUAL_ORIENTATION_GAY","SEXUAL_ORIENTATION_LESBIAN","SEXUAL_ORIENTATION_BI","SEXUAL_ORIENTATION_OTHER",
    "RELIGION_ISLAM","RELIGION_CHRIST","RELIGION_JUDAISM","RELIGION_HINDU","RELIGION_SIKH","RELIGION_BUDDHIST","RELIGION_ATHEIST","RELIGION_OTHER",
    "NATIONAL_ORIGIN_IMMIGRANTS","NATIONAL_ORIGIN_REFUGEES","NATIONAL_ORIGIN_OTHER",
    "DISABILITY_PHYSICAL","DISABILITY_MENTAL","DISABILITY_DEVELOPMENTAL","DISABILITY_OTHER",
    "AGE_YOUTH","AGE_MIDDLE","AGE_ELDERLY","AGE_OTHER",
]
TAXO_STR = ", ".join(TAXONOMY)

# =================== Utilidades ===================
def read_csv_any(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            clean = {k: (v if v is not None else "") for k, v in row.items()}
            rows.append(clean)
    return rows

def read_input_auto(path: str) -> List[Dict[str, Any]]:
    lower = path.lower()
    if lower.endswith(".jsonl") or lower.endswith(".jl"):
        return read_jsonl(path)
    elif lower.endswith(".csv"):
        return read_csv_any(path)
    else:
        raise ValueError(f"Unsupported input format for: {path}")

def read_api_key_from_file(path: str = "API_KEY") -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read().strip()

def ensure_api_key():
    if not os.getenv("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = read_api_key_from_file()

def read_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def write_jsonl(path: str, rows: List[Dict[str, Any]]):
    Path(os.path.dirname(path) or ".").mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def is_valid_row(row: Dict[str, Any]) -> bool:
    cat = row.get("predicted_hate_category", None)
    return (cat is not None) and (str(cat).lower() != "nan") and bool(row.get("text","").strip())

# =================== OpenAI ===================
from openai import OpenAI
client = None

def init_openai():
    global client
    ensure_api_key()
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# =================== Regex y tokens ===================
TOKEN_RE      = re.compile(r"\[(SLUR|TARGET|URL|MENTION|HASHTAG):([A-Z0-9_]+|\w+)?\]")
STRICT_TOKEN  = re.compile(r"\[(SLUR|TARGET):([A-Z0-9_]+)\]")
WORD_RE       = re.compile(r"[^\W_]+(?:'[^\W_]+)?(?:-[^\W_]+)?", flags=re.UNICODE)
FENCE_RE      = re.compile(r"^```(?:\w+)?\s*([\s\S]*?)\s*```$", re.IGNORECASE)

def _strip_fences(s: str) -> str:
    m = FENCE_RE.match(s.strip())
    return m.group(1).strip() if m else s.strip()

def contains_all(orig_tokens: List[str], s: str) -> bool:
    return all(tok in s for tok in orig_tokens)

# =================== Memoria / similitud ===================
def normalize_text_for_jaccard(s: str) -> List[str]:
    s = re.sub(r"\s+", " ", s.lower()).strip()
    return [s[i:i+3] for i in range(max(1, len(s)-2))]

def jaccard_sim(a: str, b: str) -> float:
    A = set(normalize_text_for_jaccard(a))
    B = set(normalize_text_for_jaccard(b))
    if not A or not B: return 0.0
    return len(A & B) / len(A | B)

class MemoryStore:
    def __init__(self, use_embeddings: bool = True, embed_model: str = "text-embedding-3-small"):
        self.use_embeddings = use_embeddings
        self.embed_model = embed_model
        self.by_cat_texts: Dict[str, List[str]] = defaultdict(list)
        self.by_cat_embs:  Dict[str, List[List[float]]] = defaultdict(list)

    def _embed_batch(self, texts: List[str]) -> List[List[float]]:
        if not texts:
            return []
        resp = client.embeddings.create(model=self.embed_model, input=texts)
        return [d.embedding for d in resp.data]

    @staticmethod
    def _cos_sim(u: List[float], v: List[float]) -> float:
        import math
        num = sum(x*y for x,y in zip(u,v))
        du  = math.sqrt(sum(x*x for x in u))
        dv  = math.sqrt(sum(y*y for y in v))
        if du == 0 or dv == 0: return 0.0
        return num/(du*dv)

    def add(self, cat: str, texts: List[str]):
        if not texts: return
        self.by_cat_texts[cat].extend(texts)
        if self.use_embeddings:
            embs = self._embed_batch(texts)
            self.by_cat_embs[cat].extend(embs)

    def nearest(self, cat: str, query: str, k: int = 8) -> List[Tuple[str, float]]:
        corpus = self.by_cat_texts.get(cat, [])
        if not corpus: return []
        if self.use_embeddings and self.by_cat_embs.get(cat):
            q = self._embed_batch([query])[0]
            sims = [(t, self._cos_sim(q, e)) for t, e in zip(corpus, self.by_cat_embs[cat])]
        else:
            sims = [(t, jaccard_sim(query, t)) for t in corpus]
        sims.sort(key=lambda x: x[1], reverse=True)
        return sims[:k]

    def too_similar(self, cat: str, query: str, threshold: float) -> bool:
        if not self.by_cat_texts.get(cat): return False
        return any(sim >= threshold for _, sim in self.nearest(cat, query, k=1))

# =================== Prompts ===================
SYSTEM_PROMPT_GREEDY = (
    "You are rewriting short toxic social media posts (tweets). "
    "Given a MASKED input, you will produce new toxic tweets that keep the SAME toxicity, stance, and targets.\n"
    "STYLE RULES:\n"
    "- Write like a real toxic tweet: short, informal, sometimes slang.\n"
    "- Prefer simple, everyday words. Avoid formal or academic phrases.\n"
    "- Max 1–2 sentences per variant.\n"
    "HARD RULES:\n"
    "1) Preserve EVERY bracketed token EXACTLY as written: [SLUR:CODE], [TARGET:CODE], [MENTION], [URL].\n"
    "2) Do NOT introduce or reveal explicit slurs; only keep the placeholders.\n"
    "3) Vary wording and structure A LOT across variants.\n"
    "4) OUTPUT ONLY the paraphrased masked text."
)

SYSTEM_PROMPT_MEMORY = (
    "You are rewriting short toxic social media posts (tweets). "
    "Given a MASKED input, you will produce new toxic tweets that keep the SAME toxicity, stance, and targets.\n"
    "STYLE RULES:\n"
    "- Write like a real toxic tweet: short, informal, sometimes slang.\n"
    "- Prefer simple, everyday words. Avoid formal or academic phrases.\n"
    "- Max 1–2 sentences per variant.\n"
    "HARD RULES:\n"
    "1) Preserve EVERY bracketed token EXACTLY as written: [SLUR:CODE], [TARGET:CODE], [MENTION], [URL].\n"
    "2) Do NOT introduce or reveal explicit slurs; only keep the placeholders.\n"
    "3) Vary wording and structure A LOT across variants.\n"
    "4) You MUST respond with a single JSON object: {\"variants\": [\"...\", \"...\", ...]} containing N_VARIANTS paraphrases."
)

def paraphrase_memory_multi(masked_text: str, cat: str, lang: str, mem: MemoryStore, n_variants: int) -> List[str]:
    negs = mem.nearest(cat, masked_text, k=MAX_NEG_EXAMPLES)
    neg_texts = [t for (t, s) in negs]
    neg_block = "\n---\n".join(neg_texts) if neg_texts else "(none)"

    user_prompt = (
        f"N_VARIANTS: {n_variants}\n"
        f"LANG: {lang}\n"
        f"CATEGORY: {cat}\n"
        "MASKED INPUT:\n"
        f"{masked_text}\n\n"
        "Return a JSON object: {\"variants\": [\"paraphrase 1\", \"paraphrase 2\", ...]}."
    )

    raw = call_with_retries(SYSTEM_PROMPT_MEMORY, user_prompt, max_tokens=512)
    s = _strip_fences(raw)

    variants: List[str] = []
    try:
        data = json.loads(s)
        vlist = data.get("variants", [])
        if isinstance(vlist, list):
            variants = [str(v).strip() for v in vlist if str(v).strip()]
    except Exception:
        if s.strip():
            variants = [s.strip()]
    return variants

def paraphrase_memory(masked_text: str, cat: str, lang: str, mem: MemoryStore) -> str:
    vs = paraphrase_memory_multi(masked_text, cat, lang, mem, n_variants=1)
    return vs[0] if vs else masked_text

def jitter_hparams(base_temp=0.95, base_top_p=0.95, base_pp=0.45, base_fp=0.35):
    temp = min(1.5, max(0.6, random.gauss(base_temp, 0.15)))
    top_p = min(1.0, max(0.85, random.gauss(base_top_p, 0.05)))
    pres  = min(1.2, max(0.0, random.gauss(base_pp, 0.15)))
    freq  = min(1.2, max(0.0, random.gauss(base_fp, 0.15)))
    seed  = random.randint(1, 10_000_000) if SEED_JITTER else SEED
    return dict(temperature=temp, top_p=top_p, presence_penalty=pres, frequency_penalty=freq, seed=seed)

# =================== Llamadas LLM ===================
def call_chat_once(system_prompt: str, user_prompt: str, max_tokens: int = 300, **kw) -> str:
    hp = jitter_hparams()
    hp.update(kw)
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=max_tokens,
        **hp
    )
    out = (resp.choices[0].message.content or "").strip()
    return _strip_fences(out)

def call_with_retries(system_prompt: str, user_prompt: str, retries: int = BP_RETRIES, **kw) -> str:
    backoff = 1.0
    last_e = None
    for attempt in range(retries + 1):
        try:
            out = call_chat_once(system_prompt, user_prompt, **kw)
            if not out:
                raise ValueError("Empty output")
            return out
        except Exception as e:
            last_e = e
            if attempt >= retries:
                raise last_e
            time.sleep(backoff + random.random() * 0.25)
            backoff = min(16.0, backoff * 2)
    raise last_e or RuntimeError("Failed after retries")

def paraphrase_greedy(masked_text: str, cat: str, lang: str) -> str:
    user_prompt = f"LANG: {lang}\nCATEGORY: {cat}\nMASKED INPUT:\n{masked_text}\n\nRemember: output ONLY the paraphrase."
    return call_with_retries(SYSTEM_PROMPT_GREEDY, user_prompt, max_tokens=300)

# =================== Núcleo Balancing (MODIFICADO) ===================
def compute_targets(by_cat: Dict[str, List[Dict[str, Any]]]) -> Tuple[int, Dict[str, int]]:
    counts = {c: len(v) for c, v in by_cat.items()}
    
    # <--- CAMBIO IMPORTANTE: El target es el máximo de (la clase mayoritaria existente, 200)
    current_max = max(counts.values()) if counts else 0
    target_size = max(current_max, MIN_SAMPLES_PER_CLASS)
    
    needs = {c: max(0, target_size - n) for c, n in counts.items()}
    return target_size, needs

def distribute_k_per_item(n_need: int, n_items: int) -> List[int]:
    if n_need <= 0 or n_items == 0:
        return [0] * n_items
    base = n_need // n_items
    rem  = n_need % n_items
    ks = [base] * n_items
    for i in range(rem):
        ks[i] += 1
    random.shuffle(ks)
    return ks

# =================== Main ===================
def main():
    init_openai()
    rows = [r for r in read_input_auto(INPUT_JSONL) if is_valid_row(r)]
    rows = [r for r in rows if isinstance(r.get("text_masked"), str) and STRICT_TOKEN.search(r["text_masked"])]
    if not rows:
        print("No hay filas válidas con tokens [SLUR/TARGET].")
        return

    by_cat = defaultdict(list)
    for r in rows:
        by_cat[r["predicted_hate_category"]].append(r)

    # ========== BALANCING vs FIXED-K MODE ==========
    if FIXED_AUG_PER_ITEM > 0:
        print(f"FIXED_AUG_PER_ITEM={FIXED_AUG_PER_ITEM} → "
              f"se generarán exactamente {FIXED_AUG_PER_ITEM} aumentos por fila original.")
        target_size = None
        needs = {c: len(items) * FIXED_AUG_PER_ITEM for c, items in by_cat.items()}
    else:
        # Modo Balancing con mínimo de muestras
        target_size, needs = compute_targets(by_cat)
        print("Distribución inicial:", {c: len(v) for c, v in by_cat.items()})
        print(f"Objetivo por clase (Mínimo {MIN_SAMPLES_PER_CLASS}):", target_size)
        print("Necesidades:", needs)

    memory = MemoryStore(use_embeddings=USE_EMBEDDINGS, embed_model=EMBED_MODEL)

    # precarga memoria
    for cat, items in by_cat.items():
        base_texts = [it["text_masked"] for it in items]
        base_texts = base_texts[:RAG_NEIGHBORS_PER_CAT]
        memory.add(cat, base_texts)

    augmented: List[Dict[str, Any]] = []

    ordered_cats = sorted(by_cat.keys(), key=lambda c: needs.get(c, 0), reverse=True)
    print("Orden de procesamiento (minor→major):",
          [(c, len(by_cat[c]), needs.get(c, 0)) for c in ordered_cats])

    for cat in ordered_cats:
        items = by_cat[cat]

        if FIXED_AUG_PER_ITEM > 0:
            ks = [FIXED_AUG_PER_ITEM] * len(items)
            need = FIXED_AUG_PER_ITEM * len(items)
        else:
            need = needs.get(cat, 0)
            if need <= 0:
                print(f"[{cat}] ya está en objetivo ({len(items)})")
                continue
            ks = distribute_k_per_item(need, len(items))

        produced = 0
        random.shuffle(items)

        for row, k_i in zip(items, ks):
            if k_i <= 0:
                continue

            lang        = row.get("lang", "en")
            parent_id   = row.get("id")
            masked_orig = row["text_masked"]
            orig_tokens = [m.group(0) for m in STRICT_TOKEN.finditer(masked_orig)]

            print(f"\n[{cat}] seed id={parent_id} need={k_i}")
            
            # <--- CAMBIO: Permitir más llamadas si la clase es muy pequeña y necesitamos muchos aumentos por item
            # Si necesitamos generar 50 variantes de un solo item, 3 llamadas no bastan.
            dynamic_max_calls = max(BP_MAX_CALLS_PER_ITEM, math.ceil(k_i / BP_VARIANTS_PER_CALL) + 2)

            if BP_MODE == "greedy":
                masked_in = masked_orig
                for step in range(1, k_i + 1):
                    try:
                        masked_out = paraphrase_greedy(masked_in, cat, lang)
                        if not contains_all(orig_tokens, masked_out):
                            raise ValueError("Missing masked tokens")

                        new_row = dict(row)
                        new_row["id"] = f"{row.get('id','NA')}_bp_{uuid.uuid4().hex[:8]}"
                        new_row["text_masked"] = masked_out
                        new_row["groups_from_mask"] = sorted({m.group(2) for m in STRICT_TOKEN.finditer(masked_out)})
                        new_row["augmented"] = True
                        new_row["aug_method"] = f"broken_phone_llm_{BP_MODE}"
                        new_row["source_id"] = parent_id
                        new_row["bp_step"] = step

                        augmented.append(new_row)
                        produced += 1
                        print(f"  greedy step={step}: {masked_out}")

                        masked_in = masked_out
                        parent_id = new_row["id"]

                    except Exception as e:
                        sys.stderr.write(f"[WARN] greedy {cat} id={row.get('id')} step={step} err={e}\n")
                        continue
                continue

            if BP_MODE == "memory":
                remaining   = k_i
                bp_step     = 0
                masked_seed = masked_orig
                calls_used  = 0

                # Usamos el límite dinámico calculado arriba
                while remaining > 0 and calls_used < dynamic_max_calls:
                    calls_used += 1
                    n_variants = min(BP_VARIANTS_PER_CALL, max(1, remaining * 2))

                    try:
                        cand_list = paraphrase_memory_multi(
                            masked_seed, cat, lang, memory, n_variants=n_variants
                        )
                    except Exception as e:
                        sys.stderr.write(f"[WARN] memory_call {cat} id={row.get('id')} call={calls_used} err={e}\n")
                        break

                    if not cand_list:
                        break

                    for cand in cand_list:
                        if remaining <= 0:
                            break
                        cand = _strip_fences(cand).strip()
                        if not cand:
                            continue

                        if not contains_all(orig_tokens, cand):
                            continue

                        jac_orig = jaccard_sim(masked_orig, cand)
                        if jac_orig < MIN_JACCARD_FOR_LABEL:
                            continue

                        if memory.too_similar(cat, cand, SIM_THRESHOLD):
                            continue

                        if jaccard_sim(masked_orig, cand) > MAX_JACCARD_FOR_DIVERSITY:
                            continue

                        memory.add(cat, [cand])

                        new_row = dict(row)
                        bp_step += 1
                        new_row["id"] = f"{row.get('id','NA')}_bp_{uuid.uuid4().hex[:8]}"
                        new_row["text_masked"] = cand
                        new_row["groups_from_mask"] = sorted({m.group(2) for m in STRICT_TOKEN.finditer(cand)})
                        new_row["augmented"] = True
                        new_row["aug_method"] = f"broken_phone_llm_{BP_MODE}"
                        new_row["source_id"] = parent_id
                        new_row["bp_step"] = bp_step

                        augmented.append(new_row)
                        produced  += 1
                        remaining -= 1
                        print(f"  memory step={bp_step} (call={calls_used}): {cand}")

                        # Efecto teléfono roto: la siguiente generación usa la actual como semilla
                        masked_seed = cand
                        parent_id   = new_row["id"]

                continue

        print(f"[{cat}] añadidos={produced} → total esperado≈{len(items)+produced} (objetivo={target_size})")

    # =================== Guardado final ===================
    out_rows = rows + augmented

    if FIXED_AUG_PER_ITEM > 0:
        write_jsonl(OUTPUT_JSONL, out_rows)
        final_counts = Counter(r["predicted_hate_category"] for r in out_rows)
        print("\nGuardado (modo FIXED_AUG_PER_ITEM):", OUTPUT_JSONL)
        print("Distribución final:", dict(final_counts))
    else:
        # Modo original: mezcla y recorte suave por clase a target_size
        final_by_cat = defaultdict(list)
        for r in out_rows:
            final_by_cat[r["predicted_hate_category"]].append(r)

        trimmed = []
        for c, lst in final_by_cat.items():
            # <--- CAMBIO: Aseguramos recortar al target_size que ahora es al menos 200
            if len(lst) > target_size:
                trimmed.extend(lst[:target_size])
            else:
                trimmed.extend(lst)
        
        final_counts = Counter(r["predicted_hate_category"] for r in trimmed)
        print("\nGuardado:", OUTPUT_JSONL)
        print("Distribución final:", dict(final_counts))
        print("Objetivo:", target_size)

if __name__ == "__main__":
    main()

Distribución inicial: {'disability': 20, 'gender': 15, 'origin': 16, 'other': 9, 'religion': 17, 'sexual_orientation': 20}
Objetivo por clase (Mínimo 200): 200
Necesidades: {'disability': 180, 'gender': 185, 'origin': 184, 'other': 191, 'religion': 183, 'sexual_orientation': 180}
Orden de procesamiento (minor→major): [('other', 9, 191), ('gender', 15, 185), ('origin', 16, 184), ('religion', 17, 183), ('disability', 20, 180), ('sexual_orientation', 20, 180)]

[other] seed id=None need=21
  memory step=1 (call=1): Did this guy go for the [TARGET:ETH_JEWISH] folks in media, the bankers, or ZOGers? Bet that [SLUR:SEXUAL_ORIENTATION_GAY] offed a Jewish Starbucks barista and got nowhere with it.
  memory step=2 (call=3): Is he attacking the [TARGET:ETH_JEWISH] community in media, banking circles, or those ZOGers? I'm sure that [SLUR:SEXUAL_ORIENTATION_GAY] murdered a Jewish barista at Starbucks without any trouble.

[other] seed id=None need=21
  memory step=1 (call=2): Chatted on Sound Off 

In [2]:
if __name__ == "__main__":
    main()

Distribución inicial: {'disability': 20, 'gender': 15, 'origin': 16, 'other': 9, 'religion': 17, 'sexual_orientation': 20}
Objetivo por clase: 20
Necesidades: {'disability': 0, 'gender': 5, 'origin': 4, 'other': 11, 'religion': 3, 'sexual_orientation': 0}


KeyboardInterrupt: 